# Decoding & Sampling — from scratch

How a language model's **next-token distribution** becomes text. This notebook builds **greedy,
temperature, top-k, and top-p (nucleus)** from scratch on a tiny hand-built distribution, and
*proves* the central claim — **top-p adapts its cutoff to the distribution's shape while top-k
cannot** — with `assert` statements, not faith.

Every function here is imported from the companion script `decoding_sampling.py`, so the numbers
in this notebook, on the concept page, and in the figures are computed in exactly one place and
cannot drift. Read the asserts as the contract: if a refactor ever broke a claim, the cell fails.

**Companion files:** the concept page `../18-Decoding-and-Sampling.md`, the runnable script
`decoding_sampling.py`, and the figure generator `make_figures_18.py` (same numbers, same palette).

## 0. Setup and device honesty

We pin the reproducible trace to **CPU** and print the device honestly — the printed device must
match where the math actually ran. The logic is device-agnostic (CUDA / MPS / CPU); only the
absolute timings would differ, and this notebook quotes *values*, not timings, so CPU is canonical.

In [1]:
import torch
import torch.nn.functional as F

# Canonical source of truth: every function and constant comes from the sibling script.
from decoding_sampling import (
    VOCAB, PEAKED_LOGITS, FLAT_LOGITS, TEMPERATURES, TOP_K, TOP_P,
    SAMPLE_SEED, N_SAMPLES,
    softmax_with_temperature, entropy_bits, greedy_token,
    top_k_filter, top_p_filter, nucleus_size,
    sample_from_logits, empirical_frequencies, DEVICE,
)

trace_device = "cpu"  # pin the trace to CPU; print the device honestly
print(f"device: {trace_device} (detected {DEVICE}; pinned to CPU for reproducibility)")
print("torch:", torch.__version__)

peaked = torch.tensor(PEAKED_LOGITS, device=trace_device)
flat = torch.tensor(FLAT_LOGITS, device=trace_device)
print("vocab:", VOCAB)

device: cpu (detected mps; pinned to CPU for reproducibility)
torch: 2.12.0
vocab: ('the', 'cat', 'sat', 'on', 'mat', 'dog', 'ran', 'fast', 'blue', 'sky')


## 1. The two distributions we'll decode from

A 10-word toy vocabulary is enough to *see* every strategy reshape the distribution. Two logit
vectors anchor everything: a **peaked** one where `cat` dominates, and a **flat** one where mass is
spread with no clear winner. The contrast between them is what makes top-p's adaptivity visible.

In [2]:
peaked_probs = F.softmax(peaked, dim=-1)
flat_probs = F.softmax(flat, dim=-1)
print("peaked:", {VOCAB[i]: round(float(p), 3) for i, p in enumerate(peaked_probs)})
print("flat:  ", {VOCAB[i]: round(float(p), 3) for i, p in enumerate(flat_probs)})
print(f"\nentropy  peaked = {entropy_bits(peaked_probs):.3f} bits   flat = {entropy_bits(flat_probs):.3f} bits")
print("(lower entropy = more peaked; max for 10 tokens is log2(10) = 3.32 bits)")

peaked: {'the': 0.045, 'cat': 0.913, 'sat': 0.017, 'on': 0.01, 'mat': 0.006, 'dog': 0.004, 'ran': 0.002, 'fast': 0.001, 'blue': 0.001, 'sky': 0.001}
flat:   {'the': 0.129, 'cat': 0.111, 'sat': 0.101, 'on': 0.117, 'mat': 0.087, 'dog': 0.106, 'ran': 0.079, 'fast': 0.096, 'blue': 0.091, 'sky': 0.083}

entropy  peaked = 0.611 bits   flat = 3.305 bits
(lower entropy = more peaked; max for 10 tokens is log2(10) = 3.32 bits)


## 2. Greedy = argmax (deterministic, myopic)

The simplest decoder: emit the single highest-probability token. Because softmax is monotonic,
the argmax of the probabilities equals the argmax of the logits — so greedy doesn't even need the
softmax. It's deterministic (no seed) but **myopic**: it optimizes one token, never the sentence,
and on open-ended text it falls into repetition loops.

In [3]:
g = greedy_token(peaked)
print(f"greedy(peaked) -> '{VOCAB[g]}' (id {g})")
assert VOCAB[g] == "cat", "greedy must pick the single highest-logit token"
print("deterministic: same input -> same token, every time (no RNG)")

greedy(peaked) -> 'cat' (id 1)
deterministic: same input -> same token, every time (no RNG)


## 3. Temperature reshapes the softmax — measured by entropy

Temperature divides the logits before the softmax: $p_i = \mathrm{softmax}(z_i / T)$. We *measure*
its effect with Shannon entropy (low = peaked, high = flat). The contract: entropy must rise
strictly as $T$ goes $0.5 \to 1.0 \to 2.0$ — hotter temperature flattens the distribution.
$T \to 0$ is the greedy limit; $T \to \infty$ is uniform.

In [4]:
entropies = {}
for temp in TEMPERATURES:
    probs = softmax_with_temperature(peaked, temp)
    h = entropy_bits(probs)
    entropies[temp] = h
    top = sorted(range(len(VOCAB)), key=lambda i: -probs[i])[:4]
    shown = "  ".join(f"{VOCAB[i]}:{probs[i]:.3f}" for i in top)
    print(f"T={temp:>3}:  H={h:.3f} bits   {shown}")

assert entropies[0.5] < entropies[1.0] < entropies[2.0], "entropy must rise with temperature"
print(f"\n=> entropy rises {entropies[0.5]:.3f} -> {entropies[1.0]:.3f} -> {entropies[2.0]:.3f} bits")

T=0.5:  H=0.032 bits   cat:0.997  the:0.002  sat:0.000  on:0.000
T=1.0:  H=0.611 bits   cat:0.913  the:0.045  sat:0.017  on:0.010
T=2.0:  H=2.203 bits   cat:0.571  the:0.127  sat:0.077  on:0.060

=> entropy rises 0.032 -> 0.611 -> 2.203 bits


## 4. top-k keeps a FIXED count; top-p keeps an ADAPTIVE count

This is the heart of the notebook. **top-k** keeps exactly $k$ tokens regardless of shape.
**top-p (nucleus)** keeps the smallest set whose cumulative probability reaches $p$ — so its
*count* is a consequence of the distribution. We assert the key result directly: the nucleus is
**smaller** on the peaked distribution and **larger** on the flat one. top-k cannot do this.

In [5]:
k_peaked = int(torch.isfinite(top_k_filter(peaked, TOP_K)).sum())
k_flat = int(torch.isfinite(top_k_filter(flat, TOP_K)).sum())
print(f"top-k  k={TOP_K}: peaked keeps {k_peaked}, flat keeps {k_flat}  (always k)")
assert k_peaked == TOP_K and k_flat == TOP_K, "top-k size is fixed regardless of shape"

n_peaked = nucleus_size(peaked, TOP_P)
n_flat = nucleus_size(flat, TOP_P)
print(f"top-p  p={TOP_P}: peaked nucleus {n_peaked}, flat nucleus {n_flat}  (ADAPTS)")
assert n_peaked < n_flat, "nucleus must be smaller when peaked than when flat (the adaptivity)"
print(f"\n=> top-p adapts: {n_peaked} (peaked) < {n_flat} (flat); top-k stuck at {TOP_K} for both")

top-k  k=3: peaked keeps 3, flat keeps 3  (always k)
top-p  p=0.9: peaked nucleus 1, flat nucleus 9  (ADAPTS)

=> top-p adapts: 1 (peaked) < 9 (flat); top-k stuck at 3 for both


## 5. The nucleus off-by-one — the bug that bites everyone

`top_p_filter` shifts the removal mask right by one and force-keeps the top-1 token. This is *not*
decoration: without the shift, the token that **crosses** the threshold is itself removed, so the
kept mass falls *just under* $p$ — and on a sharply peaked distribution where the top token already
exceeds $p$, you can remove **everything**, leaving an empty nucleus and a divide-by-zero. We verify
the kept mass is always $\ge p$ and at least one token always survives.

In [6]:
for name, logits in (("peaked", peaked), ("flat", flat)):
    filt = top_p_filter(logits, TOP_P)
    kept = torch.isfinite(filt)
    kept_mass = float(F.softmax(logits, dim=-1)[kept].sum())
    print(f"{name:>6}: nucleus has {int(kept.sum())} tokens, kept mass = {kept_mass:.3f} (>= p={TOP_P})")
    assert kept.sum() >= 1, "at least one token must always survive"
    assert kept_mass >= TOP_P - 1e-6, "kept mass must be >= p"
print("\nno empty nucleus, no divide-by-zero — the shift + force-keep-top-1 did their job")

peaked: nucleus has 1 tokens, kept mass = 0.913 (>= p=0.9)
  flat: nucleus has 9 tokens, kept mass = 0.921 (>= p=0.9)

no empty nucleus, no divide-by-zero — the shift + force-keep-top-1 did their job


## 6. Sampling RECOVERS the filtered distribution

Truncation produces a valid (renormalized) distribution; sampling from it should reproduce those
probabilities. We draw 2000 seeded samples from the flat distribution's nucleus and check the
empirical frequencies track the theoretical probabilities — and that **masked tokens are never
drawn** (their probability is exactly 0).

In [7]:
gen = torch.Generator(device=trace_device).manual_seed(SAMPLE_SEED)
filtered = top_p_filter(flat, TOP_P)
theo = F.softmax(filtered, dim=-1)
freqs = empirical_frequencies(filtered, N_SAMPLES, gen)
max_gap = float((theo - freqs).abs().max())

order = sorted(range(len(VOCAB)), key=lambda i: -theo[i])[:4]
print("theoretical:", "  ".join(f"{VOCAB[i]}:{theo[i]:.3f}" for i in order))
print("empirical:  ", "  ".join(f"{VOCAB[i]}:{freqs[i]:.3f}" for i in order))
print(f"max |theory - empirical| = {max_gap:.3f} over {N_SAMPLES} draws")
assert max_gap < 0.05, "empirical frequencies must track the theoretical probabilities"
assert (freqs[~torch.isfinite(filtered)] == 0).all(), "masked tokens must never be sampled"
print("sampling recovers the distribution; masked tokens never appear")

theoretical: the:0.141  on:0.127  cat:0.121  dog:0.115
empirical:   the:0.139  on:0.124  cat:0.129  dog:0.121
max |theory - empirical| = 0.024 over 2000 draws
sampling recovers the distribution; masked tokens never appear


## 7. Temperature widens diversity (greedy would repeat forever)

Greedy ($T\to0$) emits the *same* token every time — the repetition trap in miniature. As $T$
rises, the number of **distinct** tokens seen over many draws grows. We count distinct tokens over
2000 seeded draws at each temperature and assert diversity rises with $T$.

In [8]:
distinct = {}
for temp in TEMPERATURES:
    gen_t = torch.Generator(device=trace_device).manual_seed(SAMPLE_SEED)
    scaled = peaked / temp
    draws = [sample_from_logits(scaled, gen_t) for _ in range(N_SAMPLES)]
    distinct[temp] = len(set(draws))
    print(f"T={temp:>3}: {distinct[temp]:>2} distinct tokens in {N_SAMPLES} draws")

assert distinct[0.5] < distinct[2.0], "raising temperature must increase diversity"
print(f"\n=> diversity rises {distinct[0.5]} -> {distinct[1.0]} -> {distinct[2.0]} distinct tokens")

T=0.5:  3 distinct tokens in 2000 draws
T=1.0:  9 distinct tokens in 2000 draws
T=2.0: 10 distinct tokens in 2000 draws

=> diversity rises 3 -> 9 -> 10 distinct tokens


## 8. Recap

- **Greedy** = argmax: deterministic, myopic, loops on open-ended text.
- **Temperature** reshapes the softmax: $T<1$ sharpens (entropy 0.03 at $T{=}0.5$), $T>1$ flattens
  (entropy 2.20 at $T{=}2.0$). $T\to0$ is greedy; $T\to\infty$ is uniform.
- **top-k** keeps a fixed count (3 either way); **top-p** keeps an adaptive count — **1** token when
  peaked, **9** when flat. That adaptivity is why nucleus sampling is the open-ended default.
- Sampling **recovers** the filtered distribution; the nucleus off-by-one (keep the crossing token,
  always keep top-1) is the bug to watch for.

Everything above is computed from `decoding_sampling.py` and matches the concept page and the
generated figures exactly — one seeded source of truth.